In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

In [2]:
df = pd.read_csv("../data/train.csv")

X = df.drop(columns=['target', 'ID_code'])
y = df['target']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((160000, 200), (40000, 200))

In [3]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

from sklearn.metrics import f1_score, roc_auc_score

print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

F1: 0.4145917001338688
ROC-AUC: 0.8578445875126868


d:\6th semester study\Regrerssion-Comparison\research\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:
# Try different thresholds
thresholds = [0.3, 0.4, 0.5, 0.6]

for t in thresholds:
    y_pred_thresh = (y_prob >= t).astype(int)
    
    f1 = f1_score(y_test, y_pred_thresh)
    print(f"Threshold: {t} | F1: {f1}")

Threshold: 0.3 | F1: 0.3210018078398518
Threshold: 0.4 | F1: 0.3679644290207136
Threshold: 0.5 | F1: 0.4145917001338688
Threshold: 0.6 | F1: 0.45446936725811854


## 🧪 Preprocessing Experiments: Handling Class Imbalance

### Objective
The goal of this notebook is to improve classification performance by addressing class imbalance and analyzing the effect of decision thresholds.

---

### Dataset Characteristics
- Highly imbalanced dataset:
  - ~90% class 0
  - ~10% class 1
- This imbalance causes models to favor the majority class

---

### Baseline Observation
- Logistic Regression achieved good ROC-AUC (~0.85)
- However, F1-score was low (~0.37)
- Model failed to detect many minority class instances

---

### Approach 1: Class Weight Adjustment

- Applied `class_weight='balanced'` in Logistic Regression
- This forces the model to give more importance to minority class

#### Result:
- F1-score improved from ~0.37 → **~0.41**
- ROC-AUC remained stable (~0.85)

---

### Approach 2: Threshold Tuning

- Default threshold (0.5) is not optimal for imbalanced data
- Evaluated multiple thresholds

#### Results:

| Threshold | F1-score |
|----------|----------|
| 0.3 | 0.32 |
| 0.4 | 0.36 |
| 0.5 | 0.41 |
| 0.6 | **0.45** |

---

### Key Observations

- Increasing threshold improved F1-score
- Best performance achieved at threshold = **0.6**
- Model already learned patterns (high ROC-AUC), but needed threshold adjustment

---

### Critical Insights

- Class imbalance significantly affects model predictions
- Accuracy is not a reliable metric in this scenario
- Class weighting improves recall and F1-score
- Threshold tuning further enhances performance without retraining

---

### Conclusion

- Combining class weighting and threshold tuning leads to significant improvement
- Proper handling of imbalance is essential for classification tasks
- These techniques will be applied to more complex models in later stages